# ADAG1 — Notebook de búsqueda

Este notebook documenta la búsqueda de una configuración de prompt para el Proyecto Oráculo. La búsqueda se ejecuta únicamente sobre el lote público `busqueda`; la partición `validacion` se reserva para la comprobación final y no se usa para seleccionar candidatos.

**Método base:** ascenso por coordenadas con reinicios. Se prueban todas las opciones de una ranura, se conserva la mejor configuración y se repite hasta que no haya mejora o se alcance el máximo de pasadas. Este procedimiento es heurístico: el mejor resultado observado no se presenta como una prueba de optimalidad global.


## 1. Instalación en Google Colab

Ejecutar las celdas de instalación del notebook base `proyecto_oraculo.ipynb` o instalar las dependencias del repositorio. La ejecución completa requiere una GPU CUDA compatible, preferiblemente T4. Después de instalar Transformers y los verificadores, reiniciar la sesión de Colab.


In [ ]:
# Si se ejecuta en Colab desde una carpeta distinta, ajustar solamente esta ruta de repositorio.
# El resto del notebook usa rutas relativas al directorio actual.
from pathlib import Path
import json, random, time

ROOT = Path.cwd()
print("Directorio de trabajo:", ROOT)


In [ ]:
from ayudas import cargar_modelo, cargar_datos, dividir
from oraculo import Oraculo, espacio, RANURAS, TEMPERATURAS

MODELO = "qwen17b"
SEMILLA = 1
MAX_PASADAS = 3
REINICIOS = 2
LOTE = 16
PRESUPUESTO = 32_000_000

modelo = cargar_modelo(MODELO)
datos = cargar_datos("datos_visibles.json")
busqueda, validacion = dividir(datos)
oraculo = Oraculo(modelo, busqueda, validacion, lote=LOTE, presupuesto=PRESUPUESTO)

CONFIGS = espacio()
print("Configuraciones con temperatura 0.0:", len(CONFIGS))
print("Instancias de búsqueda:", len(busqueda))
print("Instancias de validación:", len(validacion))


## 2. Registro reproducible

Cada evaluación se guarda con configuración, precisión, semilla, reinicio, pasada, ranura, tiempo y trazas resumidas. Además, se imprime el contador acumulado, la precisión actual y la mejor precisión observada. La tabla permite distinguir la mejor configuración encontrada de una garantía matemática de optimalidad.


In [ ]:
registro = []
cache_resultados = {}
contador_evaluaciones = 0
mejor_precision_global = -1.0

# Cota de evaluaciones del procedimiento actual:
# por reinicio: 1 inicial + MAX_PASADAS * 5 ranuras * 8 opciones.
MAX_EVALUACIONES = REINICIOS * (1 + MAX_PASADAS * len(RANURAS) * 8)

def clave_config(c):
    return tuple(c.get(k) for k in ("rol", "estrategia", "formato", "estilo", "cierre", "temperatura"))

def evaluar_registrada(config, *, reinicio, pasada, ranura, opcion):
    global contador_evaluaciones, mejor_precision_global
    clave = clave_config(config)
    inicio = time.perf_counter()
    resultado = oraculo.evaluar(config, busqueda, semilla=SEMILLA)
    segundos = time.perf_counter() - inicio
    precision = float(resultado.precision)
    contador_evaluaciones += 1
    mejor_precision_global = max(mejor_precision_global, precision)
    trazas = resultado.trazas or []
    fila = {
        "evaluacion": contador_evaluaciones,
        "configuracion": dict(config),
        "precision": precision,
        "semilla": SEMILLA,
        "reinicio": reinicio,
        "pasada": pasada,
        "ranura": ranura,
        "opcion": opcion,
        "segundos": segundos,
        "n_instancias": len(busqueda),
        "primeros_fallos": [
            {"id": t.get("id"), "violo": t.get("violo")}
            for t in trazas[:10]
        ],
    }
    registro.append(fila)
    cache_resultados[clave] = resultado
    print(
        f"eval {contador_evaluaciones:3d}/{MAX_EVALUACIONES}   "
        f"actual {precision:5.1%}   mejor {mejor_precision_global:5.1%}",
        flush=True,
    )
    return resultado


## 3. Ascenso por coordenadas

En cada coordenada se comparan las opciones disponibles manteniendo fijas las demás. En caso de empate se conserva la configuración actual para evitar cambios arbitrarios. La condición de parada es que una pasada completa no mejore la precisión o que se alcance `MAX_PASADAS`.


In [ ]:
def busqueda_por_coordenadas(inicial, reinicio):
    actual = dict(inicial)
    mejor_r = evaluar_registrada(actual, reinicio=reinicio, pasada=0, ranura="inicial", opcion=None)
    mejor_precision = float(mejor_r.precision)

    for pasada in range(1, MAX_PASADAS + 1):
        mejoro = False
        for ranura in RANURAS:
            mejor_config = dict(actual)
            mejor_local = mejor_precision
            for opcion in range(8):
                candidato = dict(actual)
                candidato[ranura] = opcion
                r = evaluar_registrada(
                    candidato,
                    reinicio=reinicio,
                    pasada=pasada,
                    ranura=ranura,
                    opcion=opcion,
                )
                p = float(r.precision)
                if p > mejor_local:
                    mejor_local = p
                    mejor_config = candidato
            if mejor_local > mejor_precision:
                actual = mejor_config
                mejor_precision = mejor_local
                mejoro = True
        if not mejoro:
            break
    return actual, mejor_precision

# Dos puntos iniciales reproducibles, sin usar una configuración objetivo conocida.
iniciales = [
    {k: 0 for k in RANURAS} | {"temperatura": 0.0},
]
rng = random.Random(SEMILLA)
iniciales.append({k: rng.randrange(8) for k in RANURAS} | {"temperatura": 0.0})

resultados_reinicios = []
for i, inicial in enumerate(iniciales, start=1):
    config, precision = busqueda_por_coordenadas(inicial, reinicio=i)
    resultados_reinicios.append({"reinicio": i, "configuracion": config, "precision": precision})
    print(f"Reinicio {i}: {precision:.2%} — {config}")

mejor_observado = max(resultados_reinicios, key=lambda x: x["precision"])
MEJOR_CONFIG = mejor_observado["configuracion"]
MEJOR_PRECISION = mejor_observado["precision"]
print("\nMejor configuración observada:", MEJOR_CONFIG)
print("Precisión en búsqueda:", f"{MEJOR_PRECISION:.2%}")


## 4. Análisis de trazas

Las trazas indican la primera restricción incumplida. El siguiente resumen ayuda a justificar qué familias de errores aparecieron y qué decisiones se tomaron. La interpretación debe complementarse con observaciones concretas de la ejecución.


In [ ]:
from collections import Counter

fallos = Counter()
for fila in registro:
    for traza in fila["primeros_fallos"]:
        if traza.get("violo"):
            fallos[traza["violo"]] += 1
print("Restricciones que aparecen con mayor frecuencia en las primeras trazas:")
for nombre, cantidad in fallos.most_common(15):
    print(f"{nombre}: {cantidad}")


## 5. Validación final

La validación se realiza después de elegir `MEJOR_CONFIG`. No se usa para modificar la configuración. El resultado mide el comportamiento sobre instancias reservadas públicas; la evaluación del profesor utilizará sus datos de prueba.


In [ ]:
r_val = oraculo.validar(MEJOR_CONFIG, n=len(validacion))
VALIDACION_PRECISION = float(r_val.precision)
print("Configuración elegida:", MEJOR_CONFIG)
print("Precisión en búsqueda:", f"{MEJOR_PRECISION:.2%}")
print("Precisión en validación:", f"{VALIDACION_PRECISION:.2%}")


In [ ]:
# Guardar evidencia de búsqueda y validación.
with open("registro_busqueda.json", "w", encoding="utf-8") as f:
    json.dump({
        "modelo": MODELO,
        "semilla": SEMILLA,
        "metodo": "ascenso_por_coordenadas_con_reinicios",
        "max_pasadas": MAX_PASADAS,
        "reinicios": REINICIOS,
        "resultados_reinicios": resultados_reinicios,
        "mejor_configuracion": MEJOR_CONFIG,
        "precision_busqueda": MEJOR_PRECISION,
        "precision_validacion": VALIDACION_PRECISION,
        "evaluaciones": registro,
    }, f, ensure_ascii=False, indent=2)
print("Guardado: registro_busqueda.json")


## 6. Generación de `entrega.json`

Ejecutar esta celda únicamente después de confirmar el identificador del grupo y el número de semana académica. La configuración se toma del resultado de la búsqueda, no se escribe manualmente.


In [ ]:
from ayudas import entrega

GRUPO = "G01"
SEMANA_ACADEMICA = 0  # Completar con el número oficial del cronograma.
entrega(GRUPO, MEJOR_CONFIG, SEMANA_ACADEMICA, ruta="entrega.json")
